# HEFT operators with pmwd

Start from white noise, evolve particles with pmwd, and construct five hybrid effective field theory (HEFT) operators in memory. The cosmology and simulation settings follow the [pmwd quickstart](quickstart.ipynb).

The first example calls `simulate_heft` directly. The second saves a pmwd particle snapshot and its linear density field, reloads them, and constructs the same operators without repeating the dynamics.

In [ ]:
import os

# Optional: select a GPU before importing JAX.
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'
# Uncomment to select CPU execution.
# os.environ['JAX_PLATFORMS'] = 'cpu'

# GPU memory preallocation fraction, as in the quickstart.
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '.95'

import jax
import jax.numpy as jnp

print('JAX backend:', jax.default_backend())
print('JAX devices:', jax.devices())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from pmwd import Configuration, Cosmology, white_noise
from pmwd.heft import simulate_heft

## Configuration and initial conditions

Both quickstart configurations use a periodic box of side $256\,h^{-1}\mathrm{Mpc}$. On GPU, the particle grid is $256^3$ with spacing $1\,h^{-1}\mathrm{Mpc}$. On CPU, it is $64^3$ with spacing $4\,h^{-1}\mathrm{Mpc}$. The force mesh has twice as many cells per axis as the particle grid. For the HEFT output, we choose one cell per Lagrangian particle-grid cell.

The pmwd defaults use double precision for cosmological parameters and single precision for particle displacements and painted fields.

In [ ]:
if jax.default_backend() == 'gpu':
    ptcl_spacing = 1.            # Lagrangian space Cartesian particle grid spacing, in Mpc/h by default
    ptcl_grid_shape = (256,) * 3 
else:
    ptcl_spacing = 4.
    ptcl_grid_shape = (64,) * 3

conf = Configuration(ptcl_spacing, ptcl_grid_shape, mesh_shape=2)
output_grid = conf.ptcl_grid_shape[0]
print(f'{conf.ptcl_num:,} particles, force mesh {conf.mesh_shape}, '
      f'{conf.a_nbody_num} N-body steps, HEFT grid {(output_grid,) * 3}')

The native linear-theory path uses pmwd's Eisenstein–Hu transfer fit and integrated growth, followed by 2LPT and N-body evolution. No external transfer files are needed.

In [ ]:
cosmo = Cosmology(conf, A_s_1e9=2.0, n_s=0.96, Omega_m=0.3,
                  Omega_b=0.05, h=0.7)
seed = 0
modes = white_noise(seed, conf)
a_ref = 1.

## Construct all five fields

The same unsmoothed linear modes feed the particle dynamics and the HEFT calculation. For the HEFT weights, this example smooths the Lagrangian linear density with the field-amplitude window

$$
d(\mathbf{k}) = W_{\mathrm G}(k)\,\delta_{\mathrm L}(\mathbf{k}, a_{\mathrm{ref}}),
\qquad
W_{\mathrm G}(k) = \exp\!\left[-\frac{k^2}{2k_d^2}\right],
\qquad
k_d = 0.75\,h\,\mathrm{Mpc}^{-1}.
$$

The window is applied before taking derivatives or forming quadratic products. It affects the four density-dependent weights; it does not change the particle trajectories or the matter overdensity `O1`.

| Output | Lagrangian particle weight | Affected by this smoothing window? |
| :--- | :--- | :--- |
| `O1`: matter overdensity $\delta_m$ | $1$ for every particle | No |
| `Od`: advected linear density | $d$ | Yes |
| `OL`: advected Laplacian | $\nabla^2 d$ | Yes |
| `Od2`: advected density squared | $\mathrm{d2}=d^2-\langle d^2\rangle_q$ | Yes |
| `Os2`: advected tidal squared | $\mathrm{s2}=\sum_{ij}K_{ij}^2-\langle\sum_{ij}K_{ij}^2\rangle_q$ | Yes |

The alternative `smoothing='csst'` adopts the Lagrangian smoothing prescription of [Zhou et al. (2025), arXiv:2506.04671](https://arxiv.org/abs/2506.04671), used by [CSSTLaB](https://github.com/ShurenZhou1999/csstlab). Its field-amplitude window is

$$
W_{\mathrm{CSST}}(k) =
\exp\!\left[-\frac{R^2}{4}\bigl(\max(k-k_{\mathrm{pivot}},0)\bigr)^2\right],
\qquad
R = 5\,h^{-1}\mathrm{Mpc},
\qquad
k_{\mathrm{pivot}} = 1\,h\,\mathrm{Mpc}^{-1}.
$$

The paper defines a window on the linear **power spectrum**; the expression above is its square root, applied to the field amplitude. This option allows comparisons using the same smoothing prescription. `smoothing='none'` applies no additional Lagrangian window.

Here, the weights use the linear density extrapolated to $z=0$ (`a_ref=1`), and the particle positions are evaluated at $z=0$ (`conf.a_stop=1`). The former sets the normalization of the linear weights; the latter sets the epoch of the evolved particles.

The current [`linear_field_from_modes` implementation](../../pmwd/heft.py) requires $0<a_{\mathrm{ref}}\leq a_{\mathrm{stop}}$, because it uses the growth table computed over this simulation's time range. `simulate_heft` enforces the same restriction. This is an implementation constraint, not a general restriction on the HEFT reference epoch.

In [ ]:
operators = simulate_heft(
    modes, cosmo, conf, a_ref=a_ref, output_grid=output_grid,
    smoothing='gaussian', kd=0.75,
)
operators = jax.block_until_ready(operators)

The returned channel order is `O1`, `Od`, `OL`, `Od2`, `Os2`. Each particle carries its Lagrangian weight to its evolved position, where weighted cloud-in-cell (CIC) assignment deposits it onto the output grid.

**Why does unit weight give the matter overdensity?** For `O1`, every particle has weight $1$, so CIC first measures the local particle density. Multiplication by $N_{\mathrm{cell}}/N_{\mathrm{part}}$ makes its box mean equal to $1$, giving $\rho_{\mathrm{CIC}}/\bar\rho$. The returned field subtracts this uniform background:

$$
O_1(\mathbf{x}) = \delta_m(\mathbf{x})
= \frac{\rho_{\mathrm{CIC}}(\mathbf{x})}{\bar\rho} - 1.
$$

Equal particle weights do not imply equal cell densities: clustering changes the total weight received by each cell. The other four channels use the same global normalization, with their respective signed weights.

For the quadratic weights,

$$
\mathrm{d2}(\mathbf q) = d(\mathbf q)^2-\langle d^2\rangle_q,
\qquad
\mathrm{s2}(\mathbf q) =
\sum_{ij}K_{ij}(\mathbf q)^2
-\left\langle\sum_{ij}K_{ij}^2\right\rangle_q,
$$

where the traceless tidal tensor is

$$
K_{ij}(\mathbf k)=
\left(\frac{k_i k_j}{k^2}-\frac{\delta_{ij}}{3}\right)d(\mathbf k)
\quad (k\ne0),
\qquad K_{ij}(\mathbf0)=0.
$$

Quadratic products are computed with spectral padding and projection, including Nyquist splitting and merging. Their Lagrangian means are removed before advection. `Od2` and `Os2` are therefore advected quadratic weights, rather than squares of the final Eulerian fields.

All channels except `OL` are dimensionless; `OL` has units $(h/\mathrm{Mpc})^2$ with the default length unit. All five volume means vanish analytically. The printed means show finite-precision residuals.

In [ ]:
labels = ['O1: matter overdensity', 'Od: linear density', 'OL: Laplacian',
          'Od2: centered density squared', 'Os2: centered tidal squared']
fields = np.asarray(operators)
assert fields.shape == (5, output_grid, output_grid, output_grid)
assert fields.dtype == conf.float_dtype
print('shape:', fields.shape, 'dtype:', fields.dtype)
for label, field in zip(labels, fields):
    assert np.isfinite(field).all()
    mean = field.mean(dtype=np.float64)
    rms = np.sqrt(np.mean(field**2, dtype=np.float64))
    if field.dtype == np.float64:
        assert abs(mean) <= 2e-11
    print(f'{label:29s} mean={mean: .3e}  rms={rms:.3e}')

### Visualize the fields

Each panel shows the central $z$ slice. Its symmetric color limits are set by the 99th percentile of the absolute slice values. Values beyond those limits saturate at the end colors; the underlying arrays are unchanged.

In [ ]:
def plot_operators(fields, box_size, title):
    fig, axes = plt.subplots(1, 5, figsize=(17, 3.8), layout='constrained')
    for ax, field, label in zip(axes, fields, labels):
        section = field[:, :, field.shape[2] // 2].T
        limit = np.percentile(np.abs(section), 99)
        limit = limit if limit > 0 else 1.
        picture = ax.imshow(
            section, origin='lower', extent=(0, box_size, 0, box_size),
            cmap='RdBu_r', vmin=-limit, vmax=limit,
        )
        ax.set_title(label, fontsize=10)
        ax.set_xlabel(r'$x\ [h^{-1}\mathrm{Mpc}]$')
        fig.colorbar(picture, ax=ax, shrink=0.7, extend='both')
    axes[0].set_ylabel(r'$y\ [h^{-1}\mathrm{Mpc}]$')
    fig.suptitle(title)
    plt.show()


plot_operators(fields, conf.box_size[0], 'End-to-end HEFT fields')

## Optional final-output save

The five fields are already available as `operators`. Enable this cell to save them. Saving these final fields is separate from saving the pmwd inputs needed to construct operators with a different smoothing window.

In [ ]:
save_final = False
if save_final:
    from pathlib import Path
    destination = Path('heft_operators.npz')
    with destination.open('xb') as stream:
        np.savez_compressed(
            stream, operators=fields, labels=np.asarray(labels),
            a_ref=a_ref, a_stop=conf.a_stop, box_size=conf.box_size[0],
            smoothing='gaussian', kd=0.75,
        )
    print(f'Saved {destination}')

## Construct HEFT fields from saved pmwd outputs

An existing simulation can supply both the **unsmoothed linear density** and the **evolved particle state**. These are the inputs to `heft_operators`; the final matter-density mesh alone is insufficient, because it does not retain the particle-to-Lagrangian-label association.

This section demonstrates a complete save–load–paint sequence. The first cell prepares a small archive using the native pmwd steps. This repeats the evolution once for demonstration, because the one-call example above returns only the five fields. If the archive already exists, that cell reuses it and does not rerun pmwd. To save a different simulation, choose a different `snapshot_path`.

The archive contains:

| Entry | Meaning |
| :--- | :--- |
| `linear_field` | Unsmoothed linear density at `a_ref`, stored as a complex rFFT with `norm='forward'` |
| `pmid`, `disp` | Evolved particle coordinates, with $\mathbf x=\mathbf{pmid}\,\Delta x_{\mathrm{force}}+\mathbf{disp}$ |
| `configuration` | Geometry, units, numerical precision, and simulation settings |
| `a_ref`, `output_grid`, `fft_norm` | Linear-field reference epoch and output conventions |
| `cosmology`, `seed` | Parameters of the example that generated the snapshot |

Particle rows must retain their original C-order Lagrangian association, with the last axis changing fastest. The archive is sufficient for HEFT painting; it omits velocities and accelerations needed for restarting a simulation.

### Save a pmwd snapshot

`linear_field_from_modes` converts native pmwd Fourier coefficients to the HEFT convention and applies growth at `a_ref`. We save this field before applying any HEFT smoothing. For an existing in-memory simulation, the save block can use its `particles` and its native linear modes directly.

In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

from pmwd import Particles, boltzmann, linear_modes, lpt, nbody
from pmwd.heft import heft_operators, linear_field_from_modes

snapshot_path = Path('heft_example_data/pmwd_snapshot.npz')

if not snapshot_path.exists():
    snapshot_cosmo = boltzmann(cosmo, conf, varlin=False)
    native_modes = linear_modes(modes, snapshot_cosmo, conf)
    particles, observables = lpt(native_modes, snapshot_cosmo, conf)
    particles, _ = nbody(particles, observables, snapshot_cosmo, conf)
    linear_field = linear_field_from_modes(
        native_modes, snapshot_cosmo, conf, a_ref=a_ref,
    ).astype(jnp.promote_types(conf.float_dtype, jnp.complex64))
    jax.block_until_ready((linear_field, particles))

    configuration = asdict(conf)
    for name in ('cosmo_dtype', 'pmid_dtype', 'float_dtype'):
        configuration[name] = str(configuration[name])
    configuration = json.dumps(
        configuration, default=lambda value: np.asarray(value).tolist(),
    )
    cosmology = {
        name: float(getattr(cosmo, name))
        for name in ('A_s_1e9', 'n_s', 'Omega_m', 'Omega_b', 'h')
    }
    snapshot_path.parent.mkdir(parents=True, exist_ok=True)
    with snapshot_path.open('xb') as stream:
        np.savez(
            stream, linear_field=np.asarray(linear_field),
            pmid=np.asarray(particles.pmid), disp=np.asarray(particles.disp),
            configuration=configuration, cosmology=json.dumps(cosmology),
            seed=seed, a_ref=a_ref, output_grid=output_grid, fft_norm='forward',
        )
    print(f'Saved {snapshot_path}')
    del particles, observables, native_modes, linear_field, snapshot_cosmo
else:
    print(f'Reusing {snapshot_path}')

### Reload the snapshot

The following cells use only the saved arrays and settings. Neither the linear theory nor the particle dynamics is recomputed. The stored linear field already contains its growth normalization, so it should not be passed through `linear_field_from_modes` again.

In [ ]:
with np.load(snapshot_path, allow_pickle=False) as snapshot:
    configuration = json.loads(str(snapshot['configuration']))
    for name in ('ptcl_grid_shape', 'mesh_shape', 'growth_inistep'):
        if isinstance(configuration[name], list):
            configuration[name] = tuple(configuration[name])
    configuration['symp_splits'] = tuple(
        tuple(pair) for pair in configuration['symp_splits']
    )
    loaded_conf = Configuration(**configuration)
    loaded_linear_field = jnp.asarray(snapshot['linear_field'])
    loaded_particles = Particles(
        loaded_conf, pmid=snapshot['pmid'], disp=snapshot['disp'],
    )
    loaded_a_ref = float(snapshot['a_ref'])
    loaded_output_grid = int(snapshot['output_grid'])
    assert str(snapshot['fft_norm']) == 'forward'

print(f'Loaded {loaded_conf.ptcl_num:,} particles; '
      f'a_ref={loaded_a_ref:g}, a_stop={loaded_conf.a_stop:g}; '
      f'HEFT grid {(loaded_output_grid,) * 3}')

### Compute and plot the five fields

Use the same Gaussian window as above to reconstruct the same operators. Set `loaded_smoothing='csst'` or `'none'` to construct a different set of weights from the same saved simulation. The particle trajectories remain fixed.

In [ ]:
loaded_smoothing = 'gaussian'
loaded_operators = heft_operators(
    loaded_linear_field, loaded_particles, loaded_conf,
    output_grid=loaded_output_grid, smoothing=loaded_smoothing,
    kd=0.75, radius=5., pivot=1.,
)
loaded_operators = jax.block_until_ready(loaded_operators)
loaded_fields = np.asarray(loaded_operators)
assert loaded_fields.shape == (5, loaded_output_grid, loaded_output_grid, loaded_output_grid)
assert loaded_fields.dtype == loaded_conf.float_dtype
assert np.isfinite(loaded_fields).all()
print('shape:', loaded_fields.shape, 'dtype:', loaded_fields.dtype)

In [ ]:
plot_operators(
    loaded_fields, loaded_conf.box_size[0],
    f'HEFT fields from saved pmwd outputs ({loaded_smoothing})',
)

## Differentiation

For derivatives through the cosmological initial conditions and particle evolution, use the in-memory `simulate_heft` path. Saving and reloading arrays removes that upstream differentiation graph. The saved-snapshot example is intended for reusing a fixed simulation.

The in-memory path supports reverse-mode differentiation. Configuration and HEFT options remain static; native custom-VJP operations do not provide general forward-mode JVP support. This notebook demonstrates forward evaluation rather than a numerical gradient check.